In [ ]:
# ==========================================
# 1. SETUP & MOUNT GOOGLE DRIVE
# ==========================================
import os
import random
from google.colab import drive

drive.mount('/content/drive')

# Ensure dependencies are installed
!pip install torchaudio kagglehub -q

import torch
import torch.nn as nn
import torchaudio
import kagglehub
from IPython.display import Audio, display

# ==========================================
# 2. DEFINE THE MODEL ARCHITECTURE
# ==========================================
class CRNNAudio(nn.Module):
    def __init__(self, num_classes, input_size=64, hidden_size=256, dropout=0.3):
        super(CRNNAudio, self).__init__()
        self.cnn = nn.Sequential(
            nn.Conv1d(input_size, 128, kernel_size=3, padding=1),
            nn.BatchNorm1d(128), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(p=dropout / 2),
            nn.Conv1d(128, 256, kernel_size=3, padding=1),
            nn.BatchNorm1d(256), nn.ReLU(), nn.MaxPool1d(2), nn.Dropout(p=dropout)
        )
        self.rnn = nn.GRU(
            input_size=256, hidden_size=hidden_size, num_layers=2,
            batch_first=True, bidirectional=True, dropout=dropout
        )
        self.fc = nn.Sequential(
            nn.Dropout(p=dropout), nn.Linear(hidden_size * 2, num_classes)
        )

    def forward(self, x, input_lengths):
        x = x.transpose(1, 2)
        x = self.cnn(x)
        lengths = input_lengths // 4
        x = x.transpose(1, 2)
        packed_x = nn.utils.rnn.pack_padded_sequence(x, lengths.cpu(), batch_first=True, enforce_sorted=False)
        packed_out, _ = self.rnn(packed_x)
        out, _ = nn.utils.rnn.pad_packed_sequence(packed_out, batch_first=True)
        return self.fc(out), lengths

# ==========================================
# 3. LOAD THE BUNDLED MODEL
# ==========================================
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model_path = "/content/drive/MyDrive/audio_captcha_model.pkl"

if not os.path.exists(model_path):
    raise FileNotFoundError(f"Model not found at {model_path}. Make sure it is saved in your Drive!")

print("📥 Loading Model Bundle from Google Drive...")
checkpoint = torch.load(model_path, map_location=device)

idx_to_char = checkpoint['idx_to_char']
blank_idx = checkpoint['blank_idx']
num_classes = checkpoint['num_classes']
model_config = checkpoint['model_config']
audio_config = checkpoint['audio_config']

model = CRNNAudio(
    num_classes=num_classes,
    input_size=model_config['input_size'], hidden_size=model_config['hidden_size'], dropout=model_config['dropout']
).to(device)

model.load_state_dict(checkpoint['model_state_dict'])
model.eval()
print("✅ Fully trained Deep CRNN loaded into memory!")

# ==========================================
# 4. DOWNLOAD DATASET FOR LIVE SAMPLES
# ==========================================
print("\n🌍 Downloading samples from Kaggle...")
dataset_path = kagglehub.dataset_download("mhassansaboor/text-and-audio-captchas")
audio_dir = os.path.join(dataset_path, "Captchas", "Audio")

# Grab all valid audio files
all_audio_files = [f for f in os.listdir(audio_dir) if f.endswith(('.mp3', '.wav'))]

# ==========================================
# 5. INFERENCE LOGIC
# ==========================================
mel_transformer = torchaudio.transforms.MelSpectrogram(
    sample_rate=audio_config['sample_rate'], n_fft=audio_config['n_fft'], hop_length=audio_config['hop_length'], n_mels=audio_config['n_mels']
)
db_transformer = torchaudio.transforms.AmplitudeToDB()

def predict_single_audio(file_path):
    waveform, sr = torchaudio.load(file_path)

    # Process audio format
    if sr != audio_config['sample_rate']:
        waveform = torchaudio.functional.resample(waveform, sr, audio_config['sample_rate'])
    if waveform.shape[0] > 1:
        waveform = torch.mean(waveform, dim=0, keepdim=True)

    spec = db_transformer(mel_transformer(waveform))
    spec = spec.squeeze(0).transpose(0, 1).unsqueeze(0).to(device)

    input_len = torch.tensor([spec.shape[1]], dtype=torch.long).to(device)

    with torch.no_grad():
        logits, out_lens = model(spec, input_len)
        preds = logits.argmax(dim=2)[0][:out_lens[0]].cpu().numpy()

    # Decode string
    decoded = []
    prev_idx = -1
    for idx in preds:
        if idx != blank_idx and idx != prev_idx:
            decoded.append(idx_to_char[idx])
        prev_idx = idx

    return "".join(decoded), waveform

# ==========================================
# 6. RUN LIVE SAMPLES
# ==========================================
NUM_SAMPLES = 3
random_samples = random.sample(all_audio_files, NUM_SAMPLES)

print(f"\n🎧 Testing {NUM_SAMPLES} Random Live Samples 🎧")
print("-" * 50)

for filename in random_samples:
    actual_label = filename.split('.')[0]
    file_path = os.path.join(audio_dir, filename)

    # Predict Label
    predicted_label, wave_tensor = predict_single_audio(file_path)

    print(f"📄 File: {filename}")
    print(f"🎯 Actual Label:    {actual_label}")
    print(f"🤖 Predicted Label: {predicted_label}")

    # Play Audio in Notebook
    display(Audio(wave_tensor.numpy(), rate=audio_config['sample_rate']))
    print("-" * 50)


Mounted at /content/drive
📥 Loading Model Bundle from Google Drive...
✅ Fully trained Deep CRNN loaded into memory!

🌍 Downloading samples from Kaggle...


100%|██████████| 2.58G/2.58G [00:30<00:00, 90.9MB/s]

Extracting files...



🎧 Testing 3 Random Live Samples 🎧
--------------------------------------------------
📄 File: FB4HDG.mp3
🎯 Actual Label:    FB4HDG
🤖 Predicted Label: FB4HDG


--------------------------------------------------
📄 File: OET64U.mp3
🎯 Actual Label:    OET64U
🤖 Predicted Label: OET64U


--------------------------------------------------
📄 File: 44ZG79.mp3
🎯 Actual Label:    44ZG79
🤖 Predicted Label: 44ZG79


--------------------------------------------------
